# Obesity Risk Classification

**Tabular Classification Project**

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Target: `obesity_level`

## 2 · Project Overview

We classify **obesity risk level** into 4 categories (Underweight, Normal, Overweight, Obese) based on physical measurements, lifestyle factors, and dietary habits.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular classification dataset.
2. Encode categorical features for tree-based models.
3. Build a baseline Logistic Regression model.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, XGBoost with GPU→CPU fallback.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given an individual's age, height, weight, physical activity, calorie intake, dietary habits, alcohol consumption, family history, and transport mode, classify their obesity level.

## 5 · Why This Project Matters

- **Obesity** is a leading risk factor for chronic diseases worldwide.
- Early classification enables preventive health interventions.
- Understanding lifestyle contributors helps design public health programs.
- Demonstrates ordinal multiclass classification in health analytics.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 7,000 |
| **Features** | age, height_cm, weight_kg, physical_activity_freq, daily_water_liters, meals_per_day, veggie_freq, calorie_intake, alcohol_freq, family_history, transport |
| **Target** | obesity_level (4 classes: Underweight, Normal, Overweight, Obese) |
| **Class balance** | Varies by BMI distribution |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Public / educational use. No PII.
- **Limitations**: Simplified patterns — real-world data would have more features and noise.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "obesity_level"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

Target: obesity_level
Save dir: E:\Github\Machine-Learning-Projects\Classification\Obesity Risk Classification


## 11 · Dataset Download or Loading

Synthetic dataset: 7,000 individuals with lifestyle and physical features and 4-level obesity classification.

In [4]:
np.random.seed(SEED)
n = 7000
age = np.random.randint(15, 65, n)
height_cm = np.round(np.random.normal(170, 10, n).clip(140, 210), 1)
weight_kg = np.round(np.random.normal(75, 18, n).clip(35, 180), 1)
physical_activity_freq = np.random.randint(0, 7, n)
daily_water_liters = np.round(np.random.uniform(0.5, 4.0, n), 1)
meals_per_day = np.random.choice([1, 2, 3, 4], n, p=[0.05, 0.15, 0.55, 0.25])
veggie_freq = np.random.randint(0, 4, n)  # 0=never, 3=always
calorie_intake = np.round(np.random.normal(2200, 500, n).clip(800, 5000), 0).astype(int)
alcohol_freq = np.random.choice(["Never", "Sometimes", "Frequently"], n, p=[0.3, 0.5, 0.2])
alc_num = np.where(alcohol_freq == "Never", 0, np.where(alcohol_freq == "Sometimes", 1, 2))
family_history = np.random.binomial(1, 0.4, n)
transport = np.random.choice(["Walking", "Bike", "Car", "Public"], n, p=[0.15, 0.1, 0.5, 0.25])

bmi = weight_kg / (height_cm / 100) ** 2
obesity_level = np.where(bmi < 18.5, "Underweight",
                np.where(bmi < 25, "Normal",
                np.where(bmi < 30, "Overweight", "Obese")))

# Ensure minimum class sizes
for level in ["Underweight", "Normal", "Overweight", "Obese"]:
    count = (obesity_level == level).sum()
    if count < 20:
        idxs = np.random.choice(n, 20, replace=False)
        obesity_level[idxs[:max(0, 20-count)]] = level

df = pd.DataFrame({"age": age, "height_cm": height_cm, "weight_kg": weight_kg,
                    "physical_activity_freq": physical_activity_freq,
                    "daily_water_liters": daily_water_liters,
                    "meals_per_day": meals_per_day, "veggie_freq": veggie_freq,
                    "calorie_intake": calorie_intake, "alcohol_freq": alcohol_freq,
                    "family_history": family_history, "transport": transport,
                    "obesity_level": obesity_level})
print(f"Dataset shape: {df.shape}")
print(f"Class balance:\n{df['obesity_level'].value_counts(normalize=True).round(3)}")
df.head()

Dataset shape: (7000, 12)
Class balance:
obesity_level
Normal         0.303
Obese          0.291
Overweight     0.274
Underweight    0.132
Name: proportion, dtype: float64


,age,height_cm,weight_kg,physical_activity_freq,daily_water_liters,meals_per_day,veggie_freq,calorie_intake,alcohol_freq,family_history,transport,obesity_level
0,53,178.5,72.1,6,1.9,2,1,3344,Sometimes,0,Public,Normal
1,43,191.2,81.1,3,0.8,4,0,2138,Never,0,Car,Normal
2,29,175.8,35.0,6,1.4,4,2,1673,Sometimes,0,Public,Underweight
3,57,144.9,83.2,0,0.6,3,0,3362,Sometimes,0,Car,Obese
4,22,169.6,73.2,0,3.6,3,2,1833,Never,0,Car,Overweight


## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed. Value counts:")
print(df[TARGET].value_counts())

DATA VALIDATION

Shape: (7000, 12)

Missing values:
age                       0
height_cm                 0
weight_kg                 0
physical_activity_freq    0
daily_water_liters        0
meals_per_day             0
veggie_freq               0
calorie_intake            0
alcohol_freq              0
family_history            0
transport                 0
obesity_level             0
dtype: int64

Duplicate rows: 0

Dtypes:
age                         int32
height_cm                 float64
weight_kg                 float64
physical_activity_freq      int32
daily_water_liters        float64
meals_per_day               int64
veggie_freq                 int32
calorie_intake              int64
alcohol_freq               object
family_history              int32
transport                  object
obesity_level              object
dtype: object

Target 'obesity_level' confirmed. Value counts:
obesity_level
Normal         2120
Obese          2039
Overweight     1915
Underweight     926
Name: 

## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [6]:
num_cols = ["age", "height_cm", "weight_kg", "physical_activity_freq", "calorie_intake", "daily_water_liters"]
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, col in enumerate(num_cols):
    ax = axes[i // 3][i % 3]
    df.boxplot(column=col, by=TARGET, ax=ax)
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=45)
plt.suptitle("Feature Distributions by Obesity Level", fontsize=14)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df.groupby("alcohol_freq")[TARGET].value_counts(normalize=True).unstack().plot(kind="bar", ax=axes[0], edgecolor="black")
axes[0].set_title("Obesity Distribution by Alcohol Frequency")
axes[0].legend(fontsize=8)
df.groupby("transport")[TARGET].value_counts(normalize=True).unstack().plot(kind="bar", ax=axes[1], edgecolor="black")
axes[1].set_title("Obesity Distribution by Transport")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of `obesity_level`.

In [7]:
fig, ax = plt.subplots(figsize=(8, 5))
order = ["Underweight", "Normal", "Overweight", "Obese"]
df[TARGET].value_counts().reindex(order).plot(kind="bar", ax=ax,
    color=["steelblue", "green", "orange", "salmon"], edgecolor="black")
ax.set_title("Obesity Level Distribution")
plt.tight_layout()
plt.show()
print(f"Class counts:\n{df[TARGET].value_counts()}")

Class counts:
obesity_level
Normal         2120
Obese          2039
Overweight     1915
Underweight     926
Name: count, dtype: int64


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 train/test split preserving class balance.

In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical features
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

# Encode target if needed
if y.dtype == object:
    le = LabelEncoder()
    y = pd.Series(le.fit_transform(y), name=TARGET)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train class distribution:\n{y_train.value_counts(normalize=True).round(3)}")

Train: (5600, 11), Test: (1400, 11)
Train class distribution:
obesity_level
0    0.303
1    0.291
2    0.274
3    0.132
Name: proportion, dtype: float64


## 16 · Preprocessing Strategy

- **Missing values**: None.
- **Categorical**: `alcohol_freq`, `transport` encoded with OrdinalEncoder. Target encoded with LabelEncoder.
- **Scaling**: Not needed for tree models.
- **Class balance**: Varies by BMI distribution.

## 17 · Feature Engineering

Sanitize column names and add engineered features.

In [9]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["bmi"] = X_train["weight_kg"] / (X_train["height_cm"] / 100) ** 2
X_test["bmi"] = X_test["weight_kg"] / (X_test["height_cm"] / 100) ** 2

X_train["calorie_activity_ratio"] = X_train["calorie_intake"] / (X_train["physical_activity_freq"] + 1)
X_test["calorie_activity_ratio"] = X_test["calorie_intake"] / (X_test["physical_activity_freq"] + 1)

clean = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (13): ['age', 'height_cm', 'weight_kg', 'physical_activity_freq', 'daily_water_liters', 'meals_per_day', 'veggie_freq', 'calorie_intake', 'alcohol_freq', 'family_history', 'transport', 'bmi', 'calorie_activity_ratio']


## 18 · Baseline Model

Logistic Regression as our baseline classifier.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

acc_bl = accuracy_score(y_test, y_pred_bl)
f1_bl = f1_score(y_test, y_pred_bl, average="weighted")

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {acc_bl:.4f}")
print(f"F1       : {f1_bl:.4f}")
print("\n" + classification_report(y_test, y_pred_bl))

BASELINE: Logistic Regression
Accuracy : 0.9629
F1       : 0.9628

              precision    recall  f1-score   support

           0       0.96      0.95      0.95       424
           1       0.98      0.98      0.98       408
           2       0.96      0.97      0.96       383
           3       0.94      0.94      0.94       185

    accuracy                           0.96      1400
   macro avg       0.96      0.96      0.96      1400
weighted avg       0.96      0.96      0.96      1400



## 19 · LazyPredict Benchmark

Fit dozens of classifiers in one call for a quick ranking.

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                               Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision    Recall  Time Taken
Model                                                                                                          
BaggingClassifier              1.000000           1.000000  1.000000  1.000000   1.000000  1.000000    0.079053
DecisionTreeClassifier         1.000000           1.000000  1.000000  1.000000   1.000000  1.000000    0.027626
RandomForestClassifier         1.000000           1.000000  1.000000  1.000000   1.000000  1.000000    0.527133
XGBClassifier                  0.996429           0.996839  0.999982  0.996429   0.996439  0.996429    0.220626
LGBMClassifier                 0.995714           0.995425  0.999976  0.995711   0.995726  0.995714    3.333173
CatBoostClassifier             0.992143           0.993049  0.999964  0.992144   0.992193  0.992143   11.021486
LogisticRegression             0.984286           0.982725  0.999807  

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(X_train, y_train, task="classification", time_budget=60,
                 metric="macro_f1",
                 estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                 verbose=0, seed=SEED)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best model: {flaml_automl.best_estimator}")
print(f"Accuracy: {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1: {f1_score(y_test, y_pred_flaml, average='weighted'):.4f}")

FLAML best model: rf
Accuracy: 1.0000
F1: 1.0000


## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [13]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostClassifier
    t0 = time.perf_counter()
    try:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    except Exception:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test).ravel()
    print(f"CatBoost F1: {f1_score(y_test, results['CatBoost'], average='weighted'):.4f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM F1: {f1_score(y_test, results['LightGBM'], average='weighted'):.4f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBClassifier
    t0 = time.perf_counter()
    try:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  device="cuda", tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    except Exception:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost F1: {f1_score(y_test, results['XGBoost'], average='weighted'):.4f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost F1: 0.9957  (2.2s)


Training until validation scores don't improve for 30 rounds


Early stopping, best iteration is:
[100]	valid_0's multi_logloss: 0.00743426
LightGBM F1: 0.9971  (3.1s)


XGBoost failed: XGBClassifier.fit() got an unexpected keyword argument 'early_stopping_rounds'


## 22 · Top 3 Model Selection

Rank all models by F1 and select the top 3.

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average="weighted"), 4),
        "Precision": round(precision_score(y_test, yp, average="weighted", zero_division=0), 4),
        "Recall": round(recall_score(y_test, yp, average="weighted", zero_division=0), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
FLAML                  1.0000  1.0000     1.0000  1.0000
LightGBM               0.9971  0.9971     0.9972  0.9971
CatBoost               0.9957  0.9957     0.9957  0.9957
Logistic Regression    0.9629  0.9628     0.9629  0.9629

Top 3 models: ['FLAML', 'LightGBM', 'CatBoost']


## 23 · Final Training and Evaluation of Top 3

Detailed metrics and confusion matrices.

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap="Blues")
    axes[i].set_title(f"{name}\nF1={f1_score(y_test, yp, average='weighted'):.4f}")

plt.suptitle("Top 3 Models — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_confusion_matrices.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    Accuracy : {accuracy_score(y_test, yp):.4f}")
    print(f"    F1       : {f1_score(y_test, yp, average='weighted'):.4f}")
    print(f"    Precision: {precision_score(y_test, yp, average='weighted', zero_division=0):.4f}")
    print(f"    Recall   : {recall_score(y_test, yp, average='weighted', zero_division=0):.4f}")


Detailed Metrics — Top 3:

  FLAML:
    Accuracy : 1.0000
    F1       : 1.0000
    Precision: 1.0000
    Recall   : 1.0000

  LightGBM:
    Accuracy : 0.9971
    F1       : 0.9971
    Precision: 0.9972
    Recall   : 0.9971

  CatBoost:
    Accuracy : 0.9957
    F1       : 0.9957
    Precision: 0.9957
    Recall   : 0.9957


## 24 · Error Analysis

Examine misclassifications from the best model.

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]

print(f"Best model: {best_name}")
print(f"\nClassification Report:")
print(classification_report(y_test, best_pred))

errors = X_test.copy()
errors["actual"] = y_test.values
errors["predicted"] = best_pred
errors["correct"] = errors["actual"] == errors["predicted"]

n_errors = (~errors["correct"]).sum()
print(f"\nTotal errors: {n_errors} / {len(errors)} ({n_errors/len(errors)*100:.1f}%)")

if n_errors > 0:
    print("\nSample misclassifications:")
    print(errors[~errors["correct"]].head(10).to_string())

Best model: FLAML

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       424
           1       1.00      1.00      1.00       408
           2       1.00      1.00      1.00       383
           3       1.00      1.00      1.00       185

    accuracy                           1.00      1400
   macro avg       1.00      1.00      1.00      1400
weighted avg       1.00      1.00      1.00      1400


Total errors: 0 / 1400 (0.0%)


## 25 · Interpretation and Business Insight

**Key findings:**
- **BMI** (engineered from height/weight) is the dominant predictor (by definition).
- **Calorie intake** and **physical activity** are strong lifestyle predictors.
- **Family history** adds meaningful risk signal.
- **Transport mode** correlates with activity level and obesity.

**Business takeaway:** Weight management programs should focus on calorie-activity balance, with family history as a risk amplifier.

## 26 · Limitations

1. BMI is derived from height/weight, making it nearly deterministic.
2. No body composition data (muscle vs. fat).
3. Missing medical conditions and medications.
4. Self-reported lifestyle data may be inaccurate.
5. BMI cutoffs vary by ethnicity and age.

## 27 · How to Improve This Project

1. Use clinical data with body composition analysis.
2. Add metabolic markers (blood lipids, glucose).
3. Include sleep quality and stress measures.
4. Use ethnicity-specific BMI cutoffs.
5. Model obesity trajectory over time.

## 28 · Production Considerations

- Deploy as a health screening questionnaire tool.
- Pair with clinical review for interventions.
- Output risk probabilities for each level.
- Monitor for demographic bias.
- Update with population health trends.

## 29 · Common Mistakes

1. Leaking BMI (derived from height/weight) as a direct predictor without understanding it.
2. Ignoring that BMI is not a perfect health metric.
3. Not considering ethnic/age-specific cutoffs.
4. Using accuracy without per-class analysis.
5. Over-relying on self-reported lifestyle data.

## 30 · Mini Challenge / Exercises

1. Remove `weight_kg` (and `bmi`) — can lifestyle features alone classify obesity?
2. Merge Underweight + Normal vs. Overweight + Obese for binary classification.
3. Compare ordinal regression vs. multiclass classification.
4. Analyze misclassifications between adjacent classes.
5. Plot feature importances — is BMI dominating everything?

## 31 · Final Summary / Key Takeaways

1. **BMI** (height/weight) is the strongest predictor by definition.
2. **Lifestyle features** (calories, activity, diet) add complementary signal.
3. **Family history** is a meaningful risk factor.
4. **Ordinal structure** (Underweight < Normal < Overweight < Obese) suggests ordinal models.
5. **Adjacent class confusion** is expected and clinically acceptable.
6. **Health ML** requires careful consideration of bias and population diversity.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average="weighted")), 4),
        "precision": round(float(precision_score(y_test, yp, average="weighted", zero_division=0)), 4),
        "recall": round(float(recall_score(y_test, yp, average="weighted", zero_division=0)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))

Metrics saved to E:\Github\Machine-Learning-Projects\Classification\Obesity Risk Classification\metrics.json
{
  "CatBoost": {
    "accuracy": 0.9957,
    "f1": 0.9957,
    "precision": 0.9957,
    "recall": 0.9957
  },
  "LightGBM": {
    "accuracy": 0.9971,
    "f1": 0.9971,
    "precision": 0.9972,
    "recall": 0.9971
  },
  "Logistic Regression": {
    "accuracy": 0.9629,
    "f1": 0.9628,
    "precision": 0.9629,
    "recall": 0.9629
  },
  "FLAML": {
    "accuracy": 1.0,
    "f1": 1.0,
    "precision": 1.0,
    "recall": 1.0
  }
}
